In [1]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../../..")

from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_beus_articulatory_features

In [2]:
df = pd.read_csv("../dataset_construction/datasets/processed_beus.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])

In [3]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,ac_f,art_f,subject
0,1.47,1.53,1.53,1.60,p,l,p,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
1,1.53,1.60,1.60,1.75,l,i,p_l,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
2,1.60,1.75,1.75,1.83,i,z,p_l_i,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
3,1.83,1.97,1.97,2.16,k,ɔ,k,call,1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
4,1.97,2.16,2.16,2.17,ɔ,l,k_ɔ,call,1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
...,...,...,...,...,...,...,...,...,...,...,...,...
163314,2.23,2.32,2.32,2.39,h,i,h,he,6,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud
163315,2.39,2.53,2.53,2.67,f,ɛ,f,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud
163316,2.53,2.67,2.67,2.73,ɛ,l,f_ɛ,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud
163317,2.67,2.73,2.73,2.79,l,i,f_ɛ_l,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import VideoAutoencoder, SequenceAutoencoder

device = "mps" if torch.backends.mps.is_available() else "cpu"

art_model = VideoAutoencoder(
    input_shape=(10, 56, 112),
    input_channels=2,
    latent_dim=1024,
    use_mlp=True,
    stn_mode="separate",
    grid_sample_cpu_fallback=True,
).to(device)
art_model.load_state_dict(torch.load("../../encoder_training/ckpts/beus/art_model_ckpt.pth", map_location=device))
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)
ac_model.load_state_dict(torch.load("../../encoder_training/ckpts/beus/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (enc_conv): Sequential(
    (0): Conv1d(30, 512, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(512, 256, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
  )
  (enc_pool): AdaptiveAvgPool1d(output_size=1)
  (fc_enc): Sequential(
    (0): Linear(in_features=128, out_features=4096, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4096, out_features=2048, bias=True)
    (3): ReLU()
    (4): Linear(in_features=2048, out_features=1024, bias=True)
  )
  (fc_dec): Sequential(
    (0): Linear(in_features=1024, out_features=2048, bias=True)
    (1): ReLU()
    (2): Linear(i

In [5]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,ac_f,art_f,subject
0,1.47,1.53,1.53,1.60,p,l,p,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
1,1.53,1.60,1.60,1.75,l,i,p_l,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
2,1.60,1.75,1.75,1.83,i,z,p_l_i,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
3,1.83,1.97,1.97,2.16,k,ɔ,k,call,1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
4,1.97,2.16,2.16,2.17,ɔ,l,k_ɔ,call,1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
...,...,...,...,...,...,...,...,...,...,...,...,...
163314,2.23,2.32,2.32,2.39,h,i,h,he,6,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud
163315,2.39,2.53,2.53,2.67,f,ɛ,f,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud
163316,2.53,2.67,2.67,2.73,ɛ,l,f_ɛ,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud
163317,2.67,2.73,2.73,2.79,l,i,f_ɛ_l,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud


In [6]:
def get_vectors(art_f, ac_f, start, end, subject):
    articulatory_features = get_beus_articulatory_features(
        art_f,
        start,
        end
    )

    if articulatory_features is not None:

        articulatory_features = (
            torch.from_numpy(articulatory_features)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        art_out, art_z = art_model(
            articulatory_features,
            return_latent=True
        )
    else:
        art_out, art_z = None, None

    mfccs = get_mfccs(ac_f, start, end)

    if mfccs is not None:
        mfccs = (
            torch.from_numpy(mfccs.T)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        ac_out, ac_z = ac_model(mfccs)
    else:
        ac_out, ac_z = None, None

    return {
        "articulatory_features": (
            articulatory_features.detach().cpu().numpy()
            if articulatory_features is not None
            else None
        ),
        "art_z": (
            art_z.detach().cpu().numpy()
            if art_z is not None
            else None
        ),
        "mfccs": (
            mfccs.detach().cpu().numpy()
            if mfccs is not None
            else None
        ),
        "ac_z": (
            ac_z.detach().cpu().numpy()
            if ac_z is not None
            else None
        ),
    }

In [7]:
def distance(v1, v2):
    if v1 is None or v2 is None:
        return None
    return np.linalg.norm(v1 - v2)

In [8]:
pbar = tqdm.tqdm(df.iterrows(), total=len(df))

dists = {
    "art_latent_dist": [],
    "art_raw_dist": [],
    "ac_latent_dist": [],
    "ac_raw_dist": [],
}
for idx, data in pbar:
    
    phone1_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone1_start"], data["phone1_end"], data["subject"])
    phone2_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone2_start"], data["phone2_end"], data["subject"])

    art_latent_dist = distance(phone1_vectors["art_z"], phone2_vectors["art_z"])
    art_raw_dist = distance(phone1_vectors["articulatory_features"], phone2_vectors["articulatory_features"])

    ac_latent_dist = distance(phone1_vectors["ac_z"], phone2_vectors["ac_z"])
    ac_raw_dist = distance(phone1_vectors["mfccs"], phone2_vectors["mfccs"])

    dists["art_latent_dist"].append(art_latent_dist)
    dists["art_raw_dist"].append(art_raw_dist)
    dists["ac_latent_dist"].append(ac_latent_dist)
    dists["ac_raw_dist"].append(ac_raw_dist)

pd.DataFrame(dists).to_csv("distances_beus.csv", index=False)

100%|██████████| 163319/163319 [4:42:33<00:00,  9.63it/s]  
